# Telco Churn – Enterprise LDA
## Step 1 – Imports & Seed

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
import joblib, time, os
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
np.random.seed(42)
print('Libraries loaded')

Libraries loaded


## Step 2 – Data Loading

In [2]:
df = pd.read_csv('data/customer_churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Step 3 – Preprocessing Pipeline

In [3]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.drop(columns=['customerID'], errors='ignore')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
y = (df['Churn'] == 'Yes').astype(int)
X = df.drop('Churn', axis=1)
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer()), ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (5634, 19), Test: (1409, 19)


## Step 4 – Algorithm Pipeline Construction

In [4]:
pipeline = Pipeline([('pre', preprocessor), ('clf', LinearDiscriminantAnalysis())])
print('Pipeline:', pipeline)

Pipeline: Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer()),
                                                                  ('sc',
                                                                   StandardScaler())]),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('enc',
                                                           

## Step 5 – Baseline Dummy Benchmark

In [5]:
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
dummy_acc = accuracy_score(y_test, dummy.predict(X_test))
print(f'Dummy accuracy: {dummy_acc:.4f}')

Dummy accuracy: 0.7346


## Step 6 – Training & Convergence Monitoring

In [6]:
pipeline.fit(X_train, y_train)
print('Training complete')

Training complete


## Step 7 – Test Set Evaluation

In [7]:
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {acc:.4f}')
print(classification_report(y_test, y_pred))

Test Accuracy: 0.7857


              precision    recall  f1-score   support

           0       0.84      0.87      0.86      1035
           1       0.61      0.54      0.57       374

    accuracy                           0.79      1409
   macro avg       0.72      0.71      0.72      1409
weighted avg       0.78      0.79      0.78      1409



## Step 8 – Visualization (Confusion Matrix)

In [8]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=80)
plt.show()
print('Confusion matrix saved.')

Confusion matrix saved.


## Step 9 – Cross-Validation Stability (3-Fold)

In [9]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')
print(f'CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

CV Accuracy: 0.7981 ± 0.0043


## Step 10 – Production Serialization

In [10]:
os.makedirs('production_models', exist_ok=True)
pipeline.fit(X_train, y_train)
out_path = f'production_models/lda_churn_pipeline.joblib'
joblib.dump(pipeline, out_path, compress=3)
size_kb = os.path.getsize(out_path) / 1024
print(f'Pipeline saved to {out_path} ({size_kb:.1f} KB)')

Pipeline saved to production_models/lda_churn_pipeline.joblib (3.8 KB)


## Step 11 – Latency Smoke Test

In [11]:
X_sample = X_test.iloc[:100]
times = []
for _ in range(100):
    t0 = time.perf_counter()
    pipeline.predict(X_sample)
    times.append((time.perf_counter() - t0) * 1000)
avg_lat = np.mean(times)
print(f'Avg latency: {avg_lat:.2f} ms')
assert avg_lat < 50.0, f'Latency {avg_lat:.2f}ms exceeds 50ms SLA'
print('✅ Latency SLA passed')

Avg latency: 4.87 ms
✅ Latency SLA passed


## Step 12 – Advanced Analysis / Hyperparameter Tuning

In [12]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
results = []
for solver in ['svd', 'lsqr', 'eigen']:
    try:
        pip_tmp = Pipeline([('pre', preprocessor),
                            ('clf', LinearDiscriminantAnalysis(solver=solver))])
        sc = cross_val_score(pip_tmp, X, y, cv=3, scoring='accuracy').mean()
        results.append({'solver': solver, 'cv_accuracy': sc})
        print(f'  solver={solver}: {sc:.4f}')
    except Exception as e:
        print(f'  solver={solver} failed: {e}')
results_df = pd.DataFrame(results)
print(results_df)

  solver=svd: 0.7965
  solver=lsqr: 0.7433


  solver=eigen failed: 
All the 3 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 860, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/pipeline.py", line 645, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "/home/python/03_Custom_addons/ML/.venv/l

## Executive Decision Matrix

| Metric | Dummy Baseline | LDA Model |
|--------|---------------|----------------|
| Accuracy | (dummy_acc) | (test_acc) |
| Strategy | most_frequent | LDA |
| CV Stability | N/A | cv_mean ± cv_std |
| Latency | N/A | avg_lat ms |
| Prod Ready | ❌ | ✅ |

**Recommendation**: Deploy LDA model — outperforms baseline with acceptable latency.